In [1]:
from datasets import load_dataset
import torch
import torch.nn as nn
import pandas as pd
import numpy as pd
import matplotlib.pyplot as plt
import seaborn as sns

device = "cuda"

In [2]:
from sklearn.model_selection import train_test_split

nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name="eng-spa", split=["validation", "test"]
)
split = nmt_original_valid_set.train_test_split(train_size=0.9, seed=42)
nmt_train_set, nmt_valid_set = split["train"], split["test"]

In [3]:
nmt_train_set[0]

{'source_text': 'The two teams debated on the issue of nuclear power.',
 'target_text': 'Los dos equipos debatieron sobre el tema de la energía nuclear.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

In [4]:
import tokenizers
import tokenizers.pre_tokenizers
import tokenizers.trainers


def train_eng_spa():
    for pair in nmt_train_set:
        yield pair["source_text"]
        yield pair["target_text"]

max_length = 256
vocab_size = 10000
nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>", "<unk>", "<s>", "</s>"])
nmt_tokenizer.train_from_iterator(train_eng_spa(), nmt_tokenizer_trainer)

In [5]:
nmt_tokenizer.encode("i like soccer").ids

[72, 403, 4380]

In [6]:
nmt_tokenizer.encode("<s> Me gusta el fútbol").ids

[2, 398, 582, 220, 3402]

In [7]:
from collections import namedtuple
from torch.utils.data import DataLoader

fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device),
                       self.tgt_token_ids.to(device), self.tgt_mask.to(device))

def nmt_collate_fn(batch):
    src_texts = [pair['source_text'] for pair in batch]
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    inputs = NmtPair(src_token_ids, src_mask, tgt_token_ids[:,:-1], tgt_mask[:,:-1])
    labels = tgt_token_ids[:, 1:]
    return inputs, labels

batch_size = 32
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size, collate_fn=nmt_collate_fn,shuffle=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size, collate_fn=nmt_collate_fn)
nmt_test_loader = DataLoader(nmt_test_set, batch_size=batch_size, collate_fn=nmt_collate_fn)

In [8]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class NmtModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, pad_id=0, hidden_dim=512, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, pad_id)
        self.encoder = nn.GRU(embed_dim, hidden_dim, n_layers, batch_first=True)
        self.decoder = nn.GRU(embed_dim, hidden_dim, n_layers, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)
        src_lengths = pair.src_mask.sum(dim=1)
        src_packed = pack_padded_sequence(
            src_embeddings, lengths=src_lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden_states = self.encoder(src_packed)
        outputs, _ = self.decoder(tgt_embeddings, hidden_states)
        return self.output(outputs).permute(0,2,1)

torch.manual_seed(42)
vocab_size = nmt_tokenizer.get_vocab_size()
nmt_model = NmtModel(vocab_size).to(device)

In [9]:
import torch.optim as optim

model = NmtModel(vocab_size=vocab_size).to(device)
optimizer = optim.NAdam(model.parameters(), lr=1e-3)
xentroy = nn.CrossEntropyLoss(ignore_index=0)  # ignore padding (id=0)

def train_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0
    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        logits = model(inputs)           # (batch, vocab, seq_len)
        loss = loss_fn(logits, labels)   # labels: (batch, seq_len)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader, loss_fn):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            logits = model(inputs)
            loss = loss_fn(logits, labels)
            total_loss += loss.item()
    return total_loss / len(loader)

# training loop
n_epochs = 3
for epoch in range(n_epochs):
    train_loss = train_epoch(model, nmt_train_loader, xentroy, optimizer)
    valid_loss = eval_epoch(model, nmt_valid_loader, xentroy)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, valid_loss={valid_loss:.4f}")


Epoch 1: train_loss=3.0520, valid_loss=2.3716
Epoch 2: train_loss=2.0073, valid_loss=2.1962
Epoch 3: train_loss=1.7251, valid_loss=2.1820


In [14]:
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    output_ids = []
    for index in range(max_length):
        batch, _ = nmt_collate_fn([{"source_text": src_text, "target_text": tgt_text}])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)
            next_token_id = Y_token_ids[0, index].item()

        if next_token_id == eos_id:
            break
        output_ids.append(next_token_id)
        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token

    return nmt_tokenizer.decode(output_ids)

In [16]:
nmt_model.eval()
translate(model, "I like soccer")

'Me gusta el fútbol .'

In [ ]:
def attention(query, key, value):
    scores = query @ key.transpose(1,2) # [B,Lq,dq] @ [B,dk,Lk] = [B, Lq, Lk]
    weights = torch.softmax(scores, dim=-1)
    return weights @ value